In [78]:
# Rode esta célula apenas uma vez para instalar tudo na sua venv atual
!pip install langchain==1.4.7 langchain-core==1.7.1 pydantic==2.13.5 langchain-ollama==1.1.0 langgraph==1.2.18 ollama==0.6.3



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_ollama import ChatOllama

config_base = dict(
    model='qwen3.5:9b',
    base_url='http://localhost:11434',
    reasoning=False,
    keep_alive='30m',
    num_ctx=8192,
    num_predict=1024,
)

# Escritor: um pouco de criatividade no texto
modelo = ChatOllama(temperature=0.5, **config_base)
# Checador e Revisor: temperatura baixa = julgamento mais estável entre rodadas
modelo_juiz = ChatOllama(temperature=0.1, **config_base)

print('Modelos configurados!')


In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
import re
import time

# ==========================================
# AGENTE 1: ESCRITOR (Montar o texto)
# ==========================================

@tool
def Citation_Formatter(author: str, year: str, page: str = None) -> str:
    """Recebe os dados da fonte e formata em citação ABNT para o corpo do texto."""
    autor_fmt = author.strip().upper()
    ano_fmt = year.strip()
    if page and page.strip():
        return f"({autor_fmt}, {ano_fmt}, p. {page.strip()})"
    return f"({autor_fmt}, {ano_fmt})"

def _formatar_sobrenome_autor(nome_completo: str) -> str:
    partes = nome_completo.strip().split()
    if not partes:
        return ""
    if len(partes) == 1:
        return partes[0].upper()
    return f"{partes[-1].upper()}, {' '.join(partes[:-1])}"

@tool
def Reference_Generator(tipo_fonte: str, dados: dict) -> str:
    """Gera referências bibliográficas de fim de documento (NBR 6023) para artigo, livro ou web."""
    autores = dados.get("autores", [])
    if not autores:
        autores_fmt = "AUTOR DESCONHECIDO"
    elif isinstance(autores, list):
        if len(autores) <= 3:
            autores_fmt = "; ".join([_formatar_sobrenome_autor(a) for a in autores])
        else:
            autores_fmt = f"{_formatar_sobrenome_autor(autores[0])} et al"
    else:
        autores_fmt = _formatar_sobrenome_autor(str(autores))

    titulo = str(dados.get("titulo", "")).strip()
    ano = str(dados.get("ano", "")).strip()

    if tipo_fonte == "artigo":
        return f"{autores_fmt}. {titulo}. **{dados.get('revista', '').strip()}**, v. {dados.get('volume', '').strip()}, n. {dados.get('numero', '').strip()}, p. {dados.get('paginas', '').strip()}, {ano}."
    elif tipo_fonte == "livro":
        edicao = f"{dados.get('edicao', '').strip()}. ed. " if dados.get('edicao') else ""
        return f"{autores_fmt}. **{titulo}**. {edicao}{dados.get('local', '').strip()}: {dados.get('editora', '').strip()}, {ano}."
    elif tipo_fonte == "web":
        return f"{autores_fmt}. **{titulo}**. {ano}. Disponível em: <{dados.get('url', '').strip()}>. Acesso em: {dados.get('data_acesso', '').strip()}."

    return f"{autores_fmt}. {titulo}. {ano}."


# ==========================================
# VALIDAÇÃO ESTRUTURAL (Python puro, sem LLM)
# ==========================================

# Citação entre parênteses: (ANDERSON, 2010) / (ANDERSON JR., 2010, p. 12) / (SILVA; SOUZA, 2019)
PADRAO_PARENTESES = r"\(([A-ZÀ-Ú][^()]*?),\s?(\d{4})[^)]*\)"
# Citação narrativa: Anderson (2010) / Anderson Jr. (2010)
PADRAO_NARRATIVA = r"\b([A-ZÀ-Ú][A-Za-zÀ-ú]+(?:\s(?:Jr|JR|Filho|FILHO|Neto|NETO)\.?)?)\s\((\d{4})\)"

def normalizar_titulos(texto: str) -> str:
    """Corrige em Python o que o modelo erra sempre: '**Introdução**' ou '**# X**' viram '## X'.
    É determinístico, então não vale a pena gastar uma rodada inteira do LLM com isso."""
    linhas = []
    for linha in texto.split('\n'):
        m = re.match(r"^\s*\*\*\s*#*\s*(.{2,80}?)\s*\*\*\s*:?\s*$", linha)
        linhas.append(f"## {m.group(1)}" if m else linha)
    return '\n'.join(linhas)

def autores_citados(texto: str) -> set[str]:
    """Primeiro sobrenome de cada citação do corpo do texto, em maiúsculas."""
    corpo = re.split(r"#+\s*refer[êe]ncias", texto, flags=re.IGNORECASE)[0]
    nomes = [m[0] for m in re.findall(PADRAO_PARENTESES, corpo)] + [m[0] for m in re.findall(PADRAO_NARRATIVA, corpo)]
    return {re.split(r"[\s;,]", n.strip())[0].upper() for n in nomes}

def validar_estrutura(texto_artigo: str) -> list[str]:
    """Devolve a lista de falhas estruturais (vazia = estrutura OK).
    As mensagens dizem COMO corrigir, porque elas voltam direto para o Escritor."""
    texto_lower = texto_artigo.lower()
    erros = []

    if "introdução" not in texto_lower and "introducao" not in texto_lower:
        erros.append("Falta a seção '## Introdução'")
    if not any(x in texto_lower for x in ["conclusão", "conclusao", "considerações finais"]):
        erros.append("Falta a seção '## Conclusão'")

    titulos = [l for l in texto_artigo.split('\n') if l.strip().startswith('#')]
    if len(titulos) < 4:
        erros.append(
            f"Há só {len(titulos)} título(s) com '#'. Cada seção deve começar numa linha própria com '## ' "
            "(Introdução, 2 tópicos de desenvolvimento, Conclusão, Referências); não use negrito (**) nos títulos"
        )

    citados = autores_citados(texto_artigo)
    if not citados:
        erros.append("Nenhuma citação no corpo do texto. Use exatamente (SOBRENOME, ANO), ex.: (ANDERSON, 2010)")

    partes = re.split(r"#+\s*refer[êe]ncias", texto_artigo, flags=re.IGNORECASE)
    if len(partes) < 2:
        erros.append("Falta a seção '## Referências' no final")
    else:
        referencias = partes[1].upper()
        for autor in sorted(citados):
            if autor not in referencias:
                erros.append(f"O autor {autor} é citado no texto mas não aparece em Referências")

    return erros


In [ ]:
writer = create_agent(modelo, tools=[Citation_Formatter, Reference_Generator], system_prompt=(
    'Você é o Escritor de um artigo acadêmico CURTO (no máximo 350 palavras), em português, no padrão ABNT.\n\n'
    'FORMATO OBRIGATÓRIO (siga exatamente, cada título numa linha própria começando com #):\n'
    '# <Título do artigo>\n'
    '## Introdução\n'
    '<1 parágrafo>\n'
    '## <Título do tópico 1 do desenvolvimento>\n'
    '<1 parágrafo>\n'
    '## <Título do tópico 2 do desenvolvimento>\n'
    '<1 parágrafo>\n'
    '## Conclusão\n'
    '<1 parágrafo>\n'
    '## Referências\n'
    '<uma referência por linha>\n\n'
    'REGRAS:\n'
    '- Nunca use negrito (**) nos títulos.\n'
    '- Citações no texto SOMENTE neste formato: (SOBRENOME, ANO). Exemplo: (ANDERSON, 2010).\n'
    '- Coloque pelo menos uma citação na Introdução e uma em cada tópico do desenvolvimento.\n'
    '- Use no máximo 2 fontes: apenas livros clássicos e muito conhecidos sobre o tema, que você tem certeza que existem. '
    'Na dúvida, use 1 fonte só. Nunca invente título, autor ou ano.\n'
    '- Todo autor citado no texto precisa estar em Referências, e toda referência precisa ser citada no texto.\n'
    '- As ferramentas Citation_Formatter e Reference_Generator são opcionais; se usar, chame todas de uma vez.\n'
    '- Responda apenas com o artigo, sem comentários antes ou depois.\n\n'
    'SE RECEBER UM RASCUNHO COM AJUSTES: corrija SOMENTE os pontos listados e mantenha o resto do texto igual. '
    'Não acrescente seções, fontes ou parágrafos novos, a menos que um ajuste peça isso explicitamente.'
))

# Checador e Revisor só leem e opinam: sem ferramentas, cada um faz uma única chamada ao modelo.
# A conferência "autor citado x Referências" já é feita em Python (validar_estrutura).
sources_checker = create_agent(modelo_juiz, tools=[], system_prompt=(
    'Você é o Checador de Fontes de um artigo acadêmico curto e didático.\n'
    'Você NÃO tem acesso à internet: não tente confirmar título exato, edição, editora ou ano, '
    'e não acuse uma obra de falsa só porque o título está traduzido ou com pequenas diferenças.\n'
    'Aponte apenas problemas GRAVES e evidentes:\n'
    '- autor que claramente não tem relação com o tema;\n'
    '- obra que você tem certeza de que não existe;\n'
    '- afirmação técnica claramente errada no texto.\n'
    'Se não houver problema grave, responda exatamente: SEM PROBLEMAS GRAVES\n'
    'Caso contrário, liste no máximo 3 itens, um por linha, começando com "- ".'
))

revisor = create_agent(modelo_juiz, tools=[], system_prompt=(
    'Você é o Revisor final de um artigo acadêmico CURTO (até 350 palavras) e DIDÁTICO. '
    'Ele não precisa ser completo nem aprofundado: precisa estar correto, coerente e bem estruturado.\n\n'
    'A estrutura e o formato das citações já foram validados automaticamente: não avalie formatação.\n\n'
    'REPROVE somente por problemas BLOQUEANTES:\n'
    '- erro factual sobre o tema;\n'
    '- problema grave apontado pelo Checador;\n'
    '- texto fora do tema pedido.\n'
    'NÃO reprove por: falta de profundidade, estilo, sugestões de expansão, nível de detalhe, '
    'formatação de referência, ou qualquer coisa que seja só "poderia ser melhor".\n'
    'Se houver [AJUSTES JÁ PEDIDOS], confira se foram atendidos. Não crie exigências novas que não sejam bloqueantes.\n'
    'Se nada for bloqueante, APROVE.\n\n'
    'FORMATO DA RESPOSTA:\n'
    'Linha 1: exatamente APROVADO ou REPROVADO.\n'
    'Se REPROVADO: no máximo 3 ajustes, um por linha começando com "- ", cada um dizendo exatamente o que trocar no texto.'
))


In [ ]:
#LOOP

def veredicto(texto: str) -> str:
    """Primeira palavra da resposta, sem markdown (ex.: '**APROVADO**' -> 'APROVADO')."""
    limpo = re.sub(r'^[^A-Za-z]+', '', texto)
    primeira = limpo.split(maxsplit=1)[0] if limpo else ''
    return re.sub(r'[^A-Z]', '', primeira.upper())

def chamar(agente, conteudo: str, nome: str) -> str:
    inicio = time.perf_counter()
    resposta = agente.invoke(
        {'messages': [{'role': 'user', 'content': conteudo}]},
        config={'recursion_limit': 8}
    )
    texto = resposta['messages'][-1].content
    print(f"\n[{nome}] ({time.perf_counter() - inicio:.1f}s):\n{texto}")
    return texto

contexto = input("qual tema voce quer abordar?")
max_tentativas = 6
tentativas = 0
artigo_bruto = ""
feedback_revisor = ""
ajustes_pedidos = []   # histórico: o Revisor vê o que já pediu e não fica mudando de critério
artigo_aprovado = False

inicio_total = time.perf_counter()

while not artigo_aprovado and tentativas < max_tentativas:
    tentativas += 1
    print(f"\n--- [Rodada {tentativas}] Iniciando ciclo de produção/ajuste ---")

    # 1. Escritor monta ou ajusta o texto (recebe o rascunho anterior para não recomeçar do zero)
    conteudo_escritor = f"Tema: {contexto}"
    if feedback_revisor:
        conteudo_escritor += (
            f"\n\n[RASCUNHO ANTERIOR]:\n{artigo_bruto}"
            f"\n\n[AJUSTES OBRIGATÓRIOS - corrija só isto e mantenha o resto]:\n{feedback_revisor}"
        )
    artigo_bruto = normalizar_titulos(chamar(writer, conteudo_escritor, "Escritor"))

    # 2. Validação estrutural em Python: se falhar, nem chama Checador e Revisor
    erros = validar_estrutura(artigo_bruto)
    if erros:
        feedback_revisor = "\n".join(f"- {e}" for e in erros)
        print(f"\n[Validação] REPROVADO sem chamar o LLM:\n{feedback_revisor}")
        continue

    # 3. Checador analisa as fontes do rascunho atual
    texto_fontes_verificadas = chamar(
        sources_checker, f"Tema: {contexto}\nrascunho:\n{artigo_bruto}", "Checador"
    )

    # 4. Revisor avalia o conjunto completo
    conteudo_revisor = (
        f"Tema: {contexto}\n"
        f"rascunho:\n{artigo_bruto}\n"
        f"verificação das fontes:\n{texto_fontes_verificadas}"
    )
    if ajustes_pedidos:
        conteudo_revisor += "\n\n[AJUSTES JÁ PEDIDOS]:\n" + "\n".join(ajustes_pedidos)
    veredicto_revisor = chamar(revisor, conteudo_revisor, "Revisor")

    # 5. Critério de parada
    if veredicto(veredicto_revisor) == "APROVADO":
        artigo_aprovado = True
        print("\n=== PROCESSO CONCLUÍDO: Artigo final aprovado pelo Revisor! ===")
    else:
        feedback_revisor = veredicto_revisor.split('\n', 1)[-1].strip()
        ajustes_pedidos.append(f"Rodada {tentativas}:\n{feedback_revisor}")
        print("\n--- Artigo reprovado nesta rodada. Enviando ajustes para o Escritor... ---")

if not artigo_aprovado:
    print(f"\n=== PROCESSO INTERROMPIDO: Limite de {max_tentativas} tentativas atingido sem aprovação final. ===")
print(f"Tempo total: {time.perf_counter() - inicio_total:.1f}s em {tentativas} rodada(s)")
